In [ ]:
df_test = pd.read_excel('Deepdive_20260209.xlsx', header=None, engine='openpyxl')

for i, row in df_test.iterrows():
    if any('WBS Element' in str(val) or 'Project Definition' in str(val) for val in row):
        print(f"Headers found at row {i}:")
        print(row.tolist())
        break

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

files = {
    '2026-03-09': ('Deepdive_20260309.xlsx', 0),
    '2026-03-02': ('Deepdive_20260302.xlsx', 13),
    '2026-02-23': ('Deepdive_20260223.xlsx', 13),
    '2026-02-09': ('Deepdive_20260209.xlsx', 13),
}

dfs = []
for date, (filename, header_row) in files.items():
    try:
        df = pd.read_excel(filename, header=header_row, engine='openpyxl')
        df['Snapshot_Period'] = pd.to_datetime(date)
        df = df.dropna(subset=['Project Definition'])
        dfs.append(df)
        print(f"{filename}: {len(df)} rows loaded")
    except Exception as e:
        print(f"Error loading {filename}: {e}")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal dataset: {df_all.shape[0]} rows, {df_all.shape[1]} columns")

In [ ]:
df_test = pd.read_excel('Deepdive_20260209.xlsx', header=13, engine='openpyxl')
print("Non-null columns:")
print(df_test.notna().sum().sort_values(ascending=False).head(20))

print("\nRows with Planned Recogn. Date:")
print(df_test[df_test['Planned Recogn. Date'].notna()][
    ['Project Definition', 'WBS Element.1', 'Planned Recogn. Date', 'WBS Status']
].head(5))

In [ ]:
numeric_cols = [
    'Act cost inclu open PO', 'Act IGM %', 'Cost budget',
    'Budget IGM %', 'Material cost', 'Total hour', 'PM Hours',
    'SE hours', 'Labour cost', 'External cost', 'Open POs',
    'Quantity to be delivered', 'Material cost expected',
    'Minimum EaC costs', 'Max EaC IGM%', 'Sales',
    'Delivered Quantity', 'ordered quantity', 'ICOS per unit'
]

for col in numeric_cols:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

df_all['Cost_Overrun_Ratio'] = df_all['Act cost inclu open PO'] / df_all['Cost budget'].replace(0, np.nan)
df_all['IGM_Gap'] = df_all['Act IGM %'] - df_all['Budget IGM %']
df_all['Labour_Ratio'] = df_all['Labour cost'] / df_all['Act cost inclu open PO'].replace(0, np.nan)
df_all['External_Ratio'] = df_all['External cost'] / df_all['Act cost inclu open PO'].replace(0, np.nan)
df_all['OpenPO_Ratio'] = df_all['Open POs'] / df_all['Act cost inclu open PO'].replace(0, np.nan)
df_all['Delivery_Ratio'] = df_all['Delivered Quantity'] / df_all['ordered quantity'].replace(0, np.nan)
df_all['EaC_Budget_Ratio'] = df_all['Minimum EaC costs'] / df_all['Cost budget'].replace(0, np.nan)

df_all['Is_At_Risk'] = (df_all['IGM_Gap'] < -2).astype(int)

feature_cols = [
    'Cost_Overrun_Ratio', 'IGM_Gap', 'Labour_Ratio',
    'External_Ratio', 'OpenPO_Ratio', 'Delivery_Ratio', 'EaC_Budget_Ratio'
]

print("Is_At_Risk distribution:")
print(df_all['Is_At_Risk'].value_counts())
print(f"\nAt-risk rate: {df_all['Is_At_Risk'].mean()*100:.1f}%")
print("\nMissing data in features (%):")
print((df_all[feature_cols].isnull().sum() / len(df_all) * 100).round(1))

In [ ]:
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

df_model = df_all[feature_cols + ['Is_At_Risk']].copy()

for col in feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Clean dataset: {df_model.shape[0]} rows")
print(df_model['Is_At_Risk'].value_counts())

In [ ]:
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

df_model = df_all[feature_cols + ['Is_At_Risk']].copy()

for col in feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

df_model = df_model.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Clean dataset: {df_model.shape[0]} rows")
print(df_model['Is_At_Risk'].value_counts())

In [ ]:
df_predictions = df_all[['WBS Element.1', 'Project Definition', 'Snapshot_Period']].reset_index(drop=True)

X_full = df_model[feature_cols_clean].reset_index(drop=True)
df_predictions['Risk_Probability'] = rf.predict_proba(X_full)[:,1]
df_predictions['Is_At_Risk_Predicted'] = rf.predict(X_full)
df_predictions['Risk_Label'] = df_predictions['Risk_Probability'].apply(
    lambda x: 'High' if x > 0.7 else ('Medium' if x > 0.4 else 'Low')
)

print(df_predictions['Risk_Label'].value_counts())
df_predictions.to_csv('ML_Predictions.csv', index=False)
print("ML_Predictions.csv saved")

In [ ]:
import matplotlib.pyplot as plt

importances = pd.DataFrame({
    'Feature': feature_cols_clean,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importances['Feature'], importances['Importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - Random Forest')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()